# FINANCE 384 Assignment 1 – Part A

## A.4 Random Forest Hyperparameter Tuning

This notebook tunes the Random Forest model selected in A.3.

The required protocol is:

1. fit each candidate Random Forest on the **training sample only**;
2. evaluate each candidate on the **validation sample only**;
3. select the specification with the lowest pooled validation MSE;
4. retain the corresponding **training-fitted model**;
5. do **not** re-estimate the selected Random Forest on training + validation;
6. leave the test period untouched until final out-of-sample evaluation.

The tuning process is reproducible and uses the same revised A.1 predictor information and preprocessing framework as A.2 and A.3.


### A.4 tuning grid

The Random Forest search tunes the same core complexity dimensions used in the course material:

| Hyperparameter | Candidate values |
|---|---|
| `n_estimators` | 100 (fixed) |
| `max_depth` | 2, 3, 4 |
| `min_samples_leaf` | 2, 4, 8 |
| `max_features` | 0.3, 0.5 |

This gives **18 candidate Random Forest specifications**.

The grid is deliberately meaningful but compact enough to run on the much larger assignment panel in a reasonable execution environment.


In [ ]:
# A.4.1 Imports and fixed settings

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"

TARGET = "target_ret_excess_tp1"
RANDOM_SEED = 384

N_ESTIMATORS = 100
MAX_DEPTHS = [2, 3, 4]
MIN_SAMPLES_LEAF = [2, 4, 8]
MAX_FEATURES = [0.3, 0.5]

N_CANDIDATES = (
    len(MAX_DEPTHS)
    * len(MIN_SAMPLES_LEAF)
    * len(MAX_FEATURES)
)

print("Candidate specifications:", N_CANDIDATES)


In [ ]:
# A.4.2 Load supplied data

panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print(
    "Date range:",
    panel["date"].min().date(),
    "to",
    panel["date"].max().date(),
)
print("Unique stocks:", panel["permno"].nunique())
print(
    "Duplicate stock-month rows:",
    panel.duplicated(["permno", "date"]).sum(),
)


In [ ]:
# A.4.3 Define the revised A.1 predictor information

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [
    col for col in numeric_predictors
    if col != "down_market"
]

binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]

print("Numeric predictors:", len(numeric_predictors))


### Revised A.1 target construction

The target \(r^e_{i,t+1}\) is constructed using the next calendar month for the same stock.


In [ ]:
# A.4.4 Construct next-calendar-month excess-return target

panel = (
    panel
    .sort_values(["permno", "date"])
    .reset_index(drop=True)
)

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[
        ["permno", "month", "ret_excess_t"]
    ]
    .rename(columns={"ret_excess_t": TARGET})
    .assign(month=lambda df: df["month"] - 1)
)

panel = panel.merge(
    next_month_return,
    on=["permno", "month"],
    how="left",
    validate="one_to_one",
)

print(
    "Rows with valid next-month target:",
    panel[TARGET].notna().sum(),
)


### Revised A.1 missing-data treatment

Missing continuous characteristics are treated using contemporaneous information:

1. create a missingness indicator before imputation;
2. use the same-month FF49 industry median;
3. use the same-month market median as fallback.

The missingness indicators are retained as model inputs.


In [ ]:
# A.4.5 Missingness indicators and cross-sectional imputation

missing_characteristics = [
    col
    for col in continuous_predictors
    if panel[col].isna().any()
]

missing_indicator_columns = []

for col in missing_characteristics:
    indicator = f"{col}_was_missing"

    panel[indicator] = (
        panel[col]
        .isna()
        .astype(int)
    )

    missing_indicator_columns.append(indicator)

    industry_month_median = (
        panel
        .groupby(["month", "ff49_code"])[col]
        .transform("median")
    )

    market_month_median = (
        panel
        .groupby("month")[col]
        .transform("median")
    )

    panel[col] = (
        panel[col]
        .fillna(industry_month_median)
        .fillna(market_month_median)
    )

print(
    "Missingness indicators created:",
    len(missing_indicator_columns),
)

print(
    "Remaining missing continuous values:",
    int(
        panel[continuous_predictors]
        .isna()
        .sum()
        .sum()
    ),
)


In [ ]:
# A.4.6 Apply prescribed chronological split

analysis = panel.loc[
    panel[TARGET].notna()
].copy()

train = analysis.loc[
    (analysis["date"] >= "1990-01-01")
    & (analysis["date"] <= "2014-12-31")
].copy()

validation = analysis.loc[
    (analysis["date"] >= "2015-01-01")
    & (analysis["date"] <= "2018-12-31")
].copy()

test = analysis.loc[
    (analysis["date"] >= "2019-01-01")
    & (analysis["date"] <= "2022-11-30")
].copy()

sample_summary = pd.DataFrame({
    "Sample": ["Training", "Validation", "Test"],
    "Months": [
        train["month"].nunique(),
        validation["month"].nunique(),
        test["month"].nunique(),
    ],
    "Stock-month rows": [
        len(train),
        len(validation),
        len(test),
    ],
})

sample_summary


### Training and validation matrices

A.4 uses the training sample to estimate each candidate model and the validation sample to choose hyperparameters.

The test sample is retained only to document the fixed split. It is not transformed, predicted, scored, or used in model selection in A.4.


In [ ]:
# A.4.7 Build training and validation feature matrices

raw_feature_columns = (
    continuous_predictors
    + binary_predictors
    + missing_indicator_columns
    + categorical_predictors
)

X_train_raw = train[
    raw_feature_columns
].copy()

X_validation_raw = validation[
    raw_feature_columns
].copy()

y_train = train[
    TARGET
].to_numpy()

y_validation = validation[
    TARGET
].to_numpy()

print(
    "Training observations:",
    len(X_train_raw),
)

print(
    "Validation observations:",
    len(X_validation_raw),
)


### Common preprocessing

The same revised A.1 framework is retained:

- continuous predictors are standardised using training-sample parameters;
- `down_market` and missingness indicators are passed through;
- FF49 industry membership is one-hot encoded with one reference category omitted;
- preprocessing is fitted on training data only and applied unchanged to validation data.


In [ ]:
# A.4.8 Fit preprocessing on training only

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_predictors,
        ),
        (
            "binary",
            "passthrough",
            binary_predictors
            + missing_indicator_columns,
        ),
        (
            "industry",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_predictors,
        ),
    ],
    remainder="drop",
)

preprocessor.fit(X_train_raw)

X_train = preprocessor.transform(
    X_train_raw
)

X_validation = preprocessor.transform(
    X_validation_raw
)

feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Training matrix:",
    X_train.shape,
)

print(
    "Validation matrix:",
    X_validation.shape,
)

print(
    "Transformed predictors:",
    len(feature_names),
)


## Random Forest validation search

Every candidate below is fitted on the training sample.

For each candidate:

1. generate validation predictions;
2. calculate pooled validation MSE;
3. calculate the equivalent validation RMSE;
4. store both the fitted model and its results.

The candidate with the lowest pooled validation MSE is selected.


In [ ]:
# A.4.9 Hyperparameter search

fitted_models = []
search_rows = []

for max_depth in MAX_DEPTHS:
    for min_samples_leaf in MIN_SAMPLES_LEAF:
        for max_features in MAX_FEATURES:

            model = RandomForestRegressor(
                n_estimators=N_ESTIMATORS,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            )

            model.fit(
                X_train,
                y_train,
            )

            validation_pred = model.predict(
                X_validation
            )

            validation_mse = mean_squared_error(
                y_validation,
                validation_pred,
            )

            fitted_models.append(model)

            search_rows.append({
                "n_estimators": N_ESTIMATORS,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "max_features": max_features,
                "validation_mse": validation_mse,
                "validation_rmse": np.sqrt(validation_mse),
            })

search_results = pd.DataFrame(
    search_rows
)

print(
    "Candidate specifications evaluated:",
    len(search_results),
)


In [ ]:
# A.4.10 Rank candidate specifications

search_ranked = (
    search_results
    .sort_values(
        "validation_mse",
        ascending=True,
    )
    .reset_index(drop=True)
)

search_ranked


In [ ]:
# A.4.11 Select and retain the training-fitted winner

best_original_index = int(
    search_results[
        "validation_mse"
    ].idxmin()
)

selected_model = fitted_models[
    best_original_index
]

selected_row = (
    search_results
    .loc[[best_original_index]]
    .reset_index(drop=True)
)

selected_row


### Selected specification

The selected Random Forest is the candidate with the lowest pooled validation MSE.

Importantly, `selected_model` is the exact model object already fitted on the training sample. No training + validation refit is performed after selection.


In [ ]:
# A.4.12 Report selected hyperparameters and validation criterion

winner = selected_row.iloc[0]

print("A.4 SELECTED RANDOM FOREST")
print("-" * 50)

print(
    "n_estimators:",
    int(winner["n_estimators"]),
)

print(
    "max_depth:",
    int(winner["max_depth"]),
)

print(
    "min_samples_leaf:",
    int(winner["min_samples_leaf"]),
)

print(
    "max_features:",
    float(winner["max_features"]),
)

print(
    "\nValidation MSE:",
    float(winner["validation_mse"]),
)

print(
    "Validation RMSE:",
    float(winner["validation_rmse"]),
)


In [ ]:
# A.4.13 Protocol audit

print("A.4 PROTOCOL AUDIT")
print("-" * 50)

print(
    "Training observations:",
    len(train),
)

print(
    "Validation observations:",
    len(validation),
)

print(
    "Candidate specifications:",
    len(search_results),
)

print(
    "\nSelection criterion:",
    "Pooled validation MSE",
)

print(
    "Winner retained from training fit:",
    selected_model is fitted_models[best_original_index],
)

print(
    "Test predictions generated in A.4:",
    False,
)

print(
    "Test metric calculated in A.4:",
    False,
)

print(
    "Post-selection train+validation refit performed:",
    False,
)


## A.4 Summary

Random Forest hyperparameters are selected using pooled validation MSE.

All candidate models are fitted on the January 1990–December 2014 training sample and evaluated on the January 2015–December 2018 validation sample. The January 2019–November 2022 test period remains untouched.

The tuning grid varies `max_depth`, `min_samples_leaf`, and `max_features`, while holding `n_estimators = 100` fixed for computational practicality.

The candidate with the lowest validation MSE is retained as the selected Random Forest. In accordance with the assignment protocol, the selected model remains the original training-fitted object and is not re-estimated on the combined training and validation samples.
